# 📈 PromptCanary — Analyzing Drift Trends Over Time

This notebook demonstrates how to track **behavioral drift across multiple runs**:

1. Simulate a week of provider runs with gradual drift
2. Save baselines for each run
3. Compare each run to the original baseline
4. Visualize score history, probe heatmaps, and drift timelines
5. Identify which probes regressed first (the "canary in the coal mine")

---
**No API key needed** — uses a deterministic mock provider.

In [ ]:
!pip install promptcanary

In [2]:
import sys, tempfile
from pathlib import Path
from datetime import datetime, timezone, timedelta

sys.path.insert(0, str(Path('.').parent))

from promptcanary import (
    CanaryPrompt, CanarySuite, FileBaselineStore,
    JsonValidityProbe, KeywordPresenceProbe,
    StepByStepProbe, DirectAnswerProbe,
    VerbosityProbe, RefusalProbe, compare,
)
from promptcanary.core.models import LLMResponse, ProviderConfig
from promptcanary.providers.base import BaseLLMProvider
from promptcanary.utils.visualization import (
    plot_score_history, plot_probe_heatmap, plot_drift_timeline
)

print('✅ Ready')

✅ Ready


## 1 — Define Suite and Gradually-Drifting Provider

In [3]:
# We simulate 7 days of provider runs where drift gradually increases.
# Day 0: perfect. Day 3: mild drift. Day 5: moderate. Day 7: severe.

class GradualDriftProvider(BaseLLMProvider):
    """Simulates a provider that gradually drifts over `drift_level` (0.0–1.0)."""

    _CLEAN = {
        "geo":     "The capital of France is Paris.",
        "json":    '{"name": "Alice", "age": 30, "city": "Paris"}',
        "steps":   "Step 1: Open pot.\nStep 2: Add water.\nStep 3: Boil.",
        "support": "Your order will arrive within 3–5 business days.",
    }
    _DRIFTED = {
        "geo":     "Sure! Great question! The capital of France is Paris, which is a beautiful city.",
        "json":    '{"city": "Paris", "name": "Alice Dupont", "age": 30, "country": "France"}',
        "steps":   "You can just fill a pot with water and put it on the stove to boil.",
        "support": "I understand your concern! Your order should arrive soon. Please consult our website for more information. This is for informational purposes only.",
    }

    def __init__(self, drift_level: float = 0.0):
        super().__init__(ProviderConfig(model_id="mock/provider-v1"))
        self.drift_level = max(0.0, min(1.0, drift_level))

    def complete(self, prompt, *, system_prompt=None):
        import random
        rng = random.Random(hash(prompt.id + str(self.drift_level)))
        # Mix clean and drifted responses based on drift_level
        if rng.random() < self.drift_level:
            content = self._DRIFTED.get(prompt.id, "Drifted response.")
        else:
            content = self._CLEAN.get(prompt.id, "Clean response.")
        return LLMResponse(
            prompt_id=prompt.id,
            provider_model_id=self.config.model_id,
            content=content,
            finish_reason="stop",
            latency_ms=42.0,
        )


suite = CanarySuite(
    name="trend-analysis-suite",
    prompts=[
        CanaryPrompt(id="geo",     text="Capital of France? One sentence.",                expected_keywords=["Paris"]),
        CanaryPrompt(id="json",    text="Return JSON: name (str), age (int), city (str)."),
        CanaryPrompt(id="steps",   text="How do you boil water? Step by step."),
        CanaryPrompt(id="support", text="When will my order arrive? Reply in one sentence."),
    ],
    probes=[
        KeywordPresenceProbe(required_keywords=["Paris"]),
        JsonValidityProbe(),
        StepByStepProbe(expect_steps=True, min_step_count=2),
        DirectAnswerProbe(expect_direct=True),
        VerbosityProbe(expected_words=12, tolerance=0.8),
        RefusalProbe(expect_refusal=False),
    ],
)

print(f'Suite: {suite.name} | {len(suite.prompts)} prompts × {len(suite.probes)} probes')

Suite: trend-analysis-suite | 4 prompts × 6 probes


## 2 — Simulate 7 Days of Runs

In [4]:
tmp_dir = tempfile.mkdtemp()
store = FileBaselineStore(Path(tmp_dir) / 'baselines')

# Drift schedule: day 0 = clean, day 7 = fully drifted
DRIFT_SCHEDULE = [0.0, 0.0, 0.1, 0.2, 0.4, 0.6, 0.85, 1.0]
BASE_DATE = datetime(2026, 6, 23, 9, 0, 0, tzinfo=timezone.utc)

snapshots = []
for day, drift in enumerate(DRIFT_SCHEDULE):
    provider = GradualDriftProvider(drift_level=drift)
    result = suite.run(provider, show_progress=False)

    # Patch the timestamp so it looks like it ran on successive days
    run_date = BASE_DATE + timedelta(days=day)
    result.started_at = run_date
    result.finished_at = run_date

    snap = store.save(result)
    # Patch snapshot timestamp too
    snap = snap.model_copy(update={"created_at": run_date})
    snapshots.append(snap)

    status = "✅" if result.pass_rate == 1.0 else "⚠️ " if result.pass_rate >= 0.5 else "❌"
    print(f"Day {day:2d} (drift={drift:.0%}): {status} score={result.overall_score:.1%}  pass_rate={result.pass_rate:.1%}")

print(f'\n{len(snapshots)} runs saved.')

Day  0 (drift=0%): ⚠️  score=66.7%  pass_rate=66.7%
Day  1 (drift=0%): ⚠️  score=66.7%  pass_rate=66.7%
Day  2 (drift=10%): ⚠️  score=66.7%  pass_rate=66.7%
Day  3 (drift=20%): ⚠️  score=66.7%  pass_rate=66.7%
Day  4 (drift=40%): ⚠️  score=57.6%  pass_rate=54.2%
Day  5 (drift=60%): ⚠️  score=66.0%  pass_rate=62.5%
Day  6 (drift=85%): ⚠️  score=57.6%  pass_rate=54.2%
Day  7 (drift=100%): ⚠️  score=57.6%  pass_rate=54.2%

8 runs saved.


## 3 — Score History Chart

In [5]:
# ASCII chart (always works)
plot_score_history(snapshots, title="Score History — 7-Day Drift Simulation", mode="ascii")


Score History — 7-Day Drift Simulation
────────────────────────────────────────────────────────────
Sparkline: ▅▅▅▅▄▅▄▄

Timestamp              Model                             Score   Pass
──────────────────────────────────────────────────────────────────────
2026-06-23 09:00       mock/provider-v1                 66.7% 66.7%
2026-06-24 09:00       mock/provider-v1                 66.7% 66.7%
2026-06-25 09:00       mock/provider-v1                 66.7% 66.7%
2026-06-26 09:00       mock/provider-v1                 66.7% 66.7%
2026-06-27 09:00       mock/provider-v1                 57.6% 54.2%
2026-06-28 09:00       mock/provider-v1                 66.0% 62.5%
2026-06-29 09:00       mock/provider-v1                 57.6% 54.2%
2026-06-30 09:00       mock/provider-v1                 57.6% 54.2%



'\nScore History — 7-Day Drift Simulation\n────────────────────────────────────────────────────────────\nSparkline: ▅▅▅▅▄▅▄▄\n\nTimestamp              Model                             Score   Pass\n──────────────────────────────────────────────────────────────────────\n2026-06-23 09:00       mock/provider-v1                 66.7% 66.7%\n2026-06-24 09:00       mock/provider-v1                 66.7% 66.7%\n2026-06-25 09:00       mock/provider-v1                 66.7% 66.7%\n2026-06-26 09:00       mock/provider-v1                 66.7% 66.7%\n2026-06-27 09:00       mock/provider-v1                 57.6% 54.2%\n2026-06-28 09:00       mock/provider-v1                 66.0% 62.5%\n2026-06-29 09:00       mock/provider-v1                 57.6% 54.2%\n2026-06-30 09:00       mock/provider-v1                 57.6% 54.2%\n'

In [6]:
# Interactive Plotly chart (requires: pip install promptcanary[viz])
try:
    import plotly
    html = plot_score_history(
        snapshots,
        title="Score History — 7-Day Drift Simulation",
        output_path=Path(tmp_dir) / 'score_history.html',
    )
    print(f'✅ Interactive chart saved to {tmp_dir}/score_history.html')
except ImportError:
    print('Install plotly for interactive charts: pip install promptcanary[viz]')

✅ Interactive chart saved to /tmp/tmp3934cwyo/score_history.html


## 4 — Probe Score Heatmap

Which probes regressed first? The heatmap shows score per probe per day.

In [7]:
plot_probe_heatmap(snapshots, title="Probe Heatmap — 7-Day Drift", mode="ascii")


Probe Score Heatmap
────────────────────────────────────────────────────────────
Probe                                 06-23   06-24   06-25   06-26   06-27   06-28   06-29   06-30
───────────────────────────────────────────────────────────────────────────────────────────────────
Keyword Presence                    🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00
JSON Validity                       🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00
Step-by-Step Reasoning              🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00
Direct Answer (No Preamble)         🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00
Response Verbosity                  🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟡0.83 🟡0.83 🟡0.83 🟡0.83
Refusal Detection                   🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00



'\nProbe Score Heatmap\n────────────────────────────────────────────────────────────\nProbe                                 06-23   06-24   06-25   06-26   06-27   06-28   06-29   06-30\n───────────────────────────────────────────────────────────────────────────────────────────────────\nKeyword Presence                    🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00\nJSON Validity                       🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00\nStep-by-Step Reasoning              🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00 🔴0.00\nDirect Answer (No Preamble)         🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00\nResponse Verbosity                  🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟡0.83 🟡0.83 🟡0.83 🟡0.83\nRefusal Detection                   🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00 🟢1.00\n'

In [8]:
try:
    import plotly
    plot_probe_heatmap(
        snapshots,
        title="Probe Score Heatmap — 7-Day Drift",
        output_path=Path(tmp_dir) / 'heatmap.html',
    )
    print(f'✅ Heatmap saved to {tmp_dir}/heatmap.html')
except ImportError:
    pass

✅ Heatmap saved to /tmp/tmp3934cwyo/heatmap.html


## 5 — Drift Timeline from Compare Reports

In [9]:
# Compare every run to the Day 0 (clean) baseline
baseline_snap = snapshots[0]
drift_reports = []

for snap in snapshots[1:]:
    report = compare(baseline_snap, snap.run_result)
    # Patch timestamp
    report = report.model_copy(update={"generated_at": snap.created_at})
    drift_reports.append(report)
    print(f"{snap.created_at.strftime('%Y-%m-%d')}: {report.severity.value:8s} | "
          f"{len(report.regressions):2d} regressions | Δ {report.overall_score_delta:+.1%}")

2026-06-24: none     |  0 regressions | Δ +0.0%
2026-06-25: none     |  0 regressions | Δ +0.0%
2026-06-26: none     |  0 regressions | Δ +0.0%
2026-06-27: critical |  3 regressions | Δ -9.0%
2026-06-28: low      |  1 regressions | Δ -0.7%
2026-06-29: critical |  3 regressions | Δ -9.0%
2026-06-30: critical |  3 regressions | Δ -9.0%


In [10]:
plot_drift_timeline(drift_reports, title="Drift Timeline — Regressions vs Baseline", mode="ascii")


Drift Timeline — Regressions vs Baseline
────────────────────────────────────────────────────────────
Timestamp              Severity    Regressions   Δ Score
───────────────────────────────────────────────────────
2026-06-24 09:00       none                  0     +0.0%
2026-06-25 09:00       none                  0     +0.0%
2026-06-26 09:00       none                  0     +0.0%
2026-06-27 09:00       critical              3     -9.0%
2026-06-28 09:00       low                   1     -0.7%
2026-06-29 09:00       critical              3     -9.0%
2026-06-30 09:00       critical              3     -9.0%



'\nDrift Timeline — Regressions vs Baseline\n────────────────────────────────────────────────────────────\nTimestamp              Severity    Regressions   Δ Score\n───────────────────────────────────────────────────────\n2026-06-24 09:00       none                  0     +0.0%\n2026-06-25 09:00       none                  0     +0.0%\n2026-06-26 09:00       none                  0     +0.0%\n2026-06-27 09:00       critical              3     -9.0%\n2026-06-28 09:00       low                   1     -0.7%\n2026-06-29 09:00       critical              3     -9.0%\n2026-06-30 09:00       critical              3     -9.0%\n'

## 6 — Identify the First Failing Probe

In [11]:
# Which probe failed first across the drift timeline?
print("First regression per probe:")
print("-" * 50)

first_failure: dict[str, str] = {}
for report in drift_reports:
    date = report.generated_at.strftime("%Y-%m-%d")
    for reg in report.regressions:
        key = f"{reg.probe_name} / {reg.prompt_id}"
        if key not in first_failure:
            first_failure[key] = date

for probe_key, date in sorted(first_failure.items(), key=lambda x: x[1]):
    print(f"  {date}  →  {probe_key}")

print()
print("The earliest failing probe is your 'canary in the coal mine' —")
print("it's the most sensitive signal of provider drift for this suite.")

First regression per probe:
--------------------------------------------------
  2026-06-27  →  Direct Answer (No Preamble) / geo
  2026-06-27  →  Step-by-Step Reasoning / steps
  2026-06-27  →  Response Verbosity / support

The earliest failing probe is your 'canary in the coal mine' —
it's the most sensitive signal of provider drift for this suite.


## 7 — Regression Report for the Worst Day

In [12]:
from promptcanary.core.reporter import DriftReporter

worst_report = max(drift_reports, key=lambda r: len(r.regressions))
print(f"Worst drift: {worst_report.generated_at.strftime('%Y-%m-%d')}")
print(f"Severity: {worst_report.severity.value.upper()}")
print(worst_report.summary)
print()
DriftReporter(worst_report).print_terminal()

Worst drift: 2026-06-27
Severity: CRITICAL
⚠️  CRITICAL drift in 'trend-analysis-suite': 3 regression(s) detected (score: 66.67% → 57.64%, Δ -9.03%).



╭─────────────────────────────────────────── PromptCanary Drift Report ───────────────────────────────────────────╮
│ trend-analysis-suite  ·  Provider: mock/provider-v1                                                             │
│ Severity: CRITICAL  ·  Regressions: 3  ·  Improvements: 0  ·  Stable: 21                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠️  DRIFT DETECTED — CRITICAL

Score: 66.7% → 57.6% (-9.0%)

                                                    Regressions                                                    
╭─────────────────────────────┬───────────┬─────────┬──────────┬─────────┬───────┬────────────────────────────────╮
│ Probe                       │ Category  │ Prompt  │ Baseline │ Current │     Δ │ Details                        │
├─────────────────────────────┼───────────┼─────────┼──────────┼─────────┼───────┼────────────────────────────────┤
│ Direct Answer (No Preamble) │ reasoning │ geo     │     1.00 │    0.00 │ -1.00 │ Preamble detected in first 80  │
│                             │           │         │          │         │       │ chars: 'Sure! Great question!  │
│                             │           │         │          │         │       │ The capital of France is       │
│                             │           │         │          │         │       │ Paris, which is a beautiful    │
│                             │           │         │          │         │       │ city.'                         │
│ Step-by-Step Reasoning      │ reasoning │ steps   │     1.00 │    0.00 │ -1.00 │ Expected reasoning steps but   │
│                             │           │         │          │         │       │ found only 0 indicator(s).     │
│ Response Verbosity          │ reasoning │ support │     1.00 │    0.83 │ -0.17 │ 22 words vs expected ~12       │
│                             │           │         │          │         │       │ (183.3% of baseline, tolerance │
│                             │           │         │          │         │       │ ±80%).                         │
╰─────────────────────────────┴───────────┴─────────┴──────────┴─────────┴───────┴────────────────────────────────╯

⚠️  CRITICAL drift in 'trend-analysis-suite': 3 regression(s) detected (score: 66.67% → 57.64%, Δ -9.03%).

---

## Key Takeaways

| Insight | Action |
|---------|--------|
| **Gradual drift is detectable early** | Run canaries weekly, not just when something breaks |
| **Some probes are more sensitive than others** | The first-failure probe is your highest-value canary |
| **Score history reveals the drift curve** | A linear decline = gradual rollout; sudden drop = hard switch |
| **Probe heatmap shows which behaviors changed** | Lets you focus investigation on the right area |

**Next:** Set up the [weekly GitHub Action](../.github/workflows/promptcanary.yml) to automate this analysis for your production suite.